# Carnatify — raga feature extraction on Colab Pro (GPU)

Runs the production-matched Demucs+pyin pipeline (`raga_v2_pipeline.py`) over the
shankarkrish/archive.org concert downloads, so the raga classifier can be retrained
on features that match live inference exactly.

**Before running:**
1. Runtime → Change runtime type → **GPU** (any; Demucs is the GPU step).
2. On your Mac, from the repo root: `cd data && zip -r ~/carnatify_concert_audio.zip concert_audio`
   and upload `carnatify_concert_audio.zip` to your Google Drive root (My Drive).
3. Run all cells. Features accumulate in `MyDrive/carnatify_raga_cache/` — one `.npz`
   per track — so the run is **resumable across Colab sessions** (already-cached
   tracks are skipped).

**After it finishes**, back on your Mac:
```bash
# copy the Drive cache folder into the repo, then retrain + evaluate
cp -r ~/Google\ Drive/.../carnatify_raga_cache/* data/raga_v2_cache/archive/
./venv_train/bin/python train_raga_v2_archive.py
./venv_train/bin/python train_raga_v2_evaluate.py
```

Time estimate: Demucs on GPU is fast (~3-5 s/track) but `librosa.pyin` runs on CPU
three times per track (original + 2 augmentations) — expect roughly 45-90 s/track
with 4 workers, i.e. a long session (or two) for ~1,500 tracks. It is safe to stop
and re-run anytime.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip -q install demucs soundfile librosa
!git clone -q https://github.com/ShyamRavidath/carnatify.git /content/carnatify
!unzip -q -n /content/drive/MyDrive/carnatify_concert_audio.zip -d /content/
!ls /content/concert_audio | head

In [ ]:
import sys
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

sys.path.insert(0, '/content/carnatify')
from raga_v2_pipeline import process_track

AUDIO_ROOT = Path('/content/concert_audio')
CACHE_DIR = Path('/content/drive/MyDrive/carnatify_raga_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

jobs = []
for raga_dir in sorted(AUDIO_ROOT.iterdir()):
    if not raga_dir.is_dir():
        continue
    for mp3 in sorted(raga_dir.glob('*.mp3')):
        # must match train_raga_v2_archive.py's naming so local + Colab caches merge
        track_id = f'archive__{raga_dir.name}__{mp3.stem}'
        jobs.append((track_id, raga_dir.name, str(mp3)))

already = {p.stem for p in CACHE_DIR.glob('*.npz')}
todo = [j for j in jobs if j[0] not in already]
print(f'{len(jobs)} tracks total, {len(already)} cached, {len(todo)} to process')

def run(job):
    track_id, raga, path = job
    try:
        return track_id, process_track(track_id, raga, path, CACHE_DIR)
    except Exception as exc:
        return track_id, {'status': f'error: {exc}'}

done = ok = 0
with ProcessPoolExecutor(max_workers=4) as pool:
    for track_id, result in map(lambda f: f.result(), as_completed([pool.submit(run, j) for j in todo])):
        done += 1
        status = (result or {}).get('status', 'skipped (too short / no pitch)')
        if status in ('ok', 'cached'):
            ok += 1
        if status.startswith('error') or done % 10 == 0:
            print(f'[{done}/{len(todo)}] {track_id}: {status}', flush=True)
print(f'\nfinished: {ok} ok of {done} processed; cache now has {len(list(CACHE_DIR.glob("*.npz")))} tracks')